# 🔍 FleetLogix — Avance 2: Análisis y Optimización SQL

---

## ¿Qué hace este notebook?

Ejecuta **12 queries SQL** sobre la base de datos FleetLogix (505,000+ registros),
mide su performance con `EXPLAIN ANALYZE`, aplica **5 índices de optimización**
provistos por la escuela y documenta la mejora obtenida.

| Sección | Contenido |
|---------|-----------|
| **Parte 1** | Configuración y conexión a PostgreSQL |
| **Parte 2** | 12 queries organizadas por nivel (básico / intermedio / complejo) |
| **Parte 3** | EXPLAIN ANALYZE — tiempos ANTES de los índices |
| **Parte 4** | Creación de los 5 índices de optimización |
| **Parte 5** | EXPLAIN ANALYZE — tiempos DESPUÉS de los índices |
| **Parte 6** | Reporte comparativo de mejora (%) |

## Niveles de las 12 queries

| # | Nombre | Nivel | Conceptos clave |
|---|--------|-------|-----------------|
| B1 | Vehículos y último mantenimiento | Básico | LEFT JOIN, MAX, GROUP BY |
| B2 | Rutas con más viajes y combustible | Básico | JOIN, WHERE, SUM, AVG |
| B3 | Distribución de estados de entrega | Básico | GROUP BY, porcentaje con OVER() |
| I1 | Conductores más productivos | Intermedio | 3-table JOIN, HAVING, COUNT DISTINCT |
| I2 | Eficiencia por tipo de vehículo | Intermedio | LEFT JOINs múltiples, NULLIF |
| I3 | Reporte mensual de operaciones | Intermedio | EXTRACT, CASE WHEN |
| I4 | Tiempo real vs estimado por ruta | Intermedio | EPOCH, HAVING, subtracción de timestamps |
| I5 | Vehículos con mantenimiento sobre promedio | Intermedio | Subconsulta anidada en HAVING |
| C1 | Ranking de conductores por eficiencia | Complejo | CTE doble, RANK() |
| C2 | Tendencia mensual con variación % | Complejo | LAG(), promedio móvil, PARTITION BY |
| C3 | Clasificación de rutas por cuartil | Complejo | NTILE(4), CASE WHEN multivariable |
| C4 | Análisis de fatiga de conductores | Complejo | LAG() por conductor, riesgo operacional |

---



## Parte 1 — Configuración y conexión


In [1]:
import psycopg2
import pandas as pd
import time
import json
from datetime import datetime
from IPython.display import display

# ── Configuración de conexión ─────────────────────────────────
DB_CONFIG = {
    'host':     'localhost',
    'database': 'postgres',
    'user':     'postgres',
    'password': 'TU_PASSWORD',   # ← CAMBIA ESTO
    'port':     5432
}

def get_conn():
    """Abre y devuelve una conexión nueva a PostgreSQL."""
    return psycopg2.connect(**DB_CONFIG)

def run_query(sql, label=''):
    """
    Ejecuta una query SELECT y devuelve un DataFrame.
    También imprime cuántas filas devolvió y en cuánto tiempo.
    """
    conn = get_conn()
    t0   = time.time()
    df   = pd.read_sql_query(sql, conn)
    ms   = round((time.time() - t0) * 1000, 1)
    conn.close()
    if label:
        print(f'  ✅ {label} — {len(df)} filas — {ms} ms')
    return df

def run_explain(sql, label=''):
    """
    Ejecuta EXPLAIN ANALYZE sobre una query y devuelve
    el tiempo de ejecución en ms que reporta PostgreSQL.
    Imprime el plan completo para analizar.
    """
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute(f'EXPLAIN ANALYZE {sql}')
    plan = cur.fetchall()
    conn.close()

    plan_text = '\n'.join(row[0] for row in plan)
    # Extraer el tiempo de ejecución de la última línea del plan
    # PostgreSQL siempre termina con: "Execution Time: X.XXX ms"
    exec_time = None
    for row in plan:
        if 'Execution Time' in row[0]:
            # Extraer el número: "Execution Time: 123.456 ms"
            exec_time = float(row[0].split(':')[1].strip().replace(' ms',''))

    print(f'\n── EXPLAIN ANALYZE: {label} ──')
    print(plan_text)
    print(f'\n  ⏱  Execution Time: {exec_time} ms')
    return exec_time

# Verificar conexión
try:
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT table_name, 
               (SELECT COUNT(*) FROM information_schema.columns 
                WHERE table_name = t.table_name 
                AND table_schema = 'public') AS cols
        FROM information_schema.tables t
        WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
        ORDER BY table_name
    """)
    tablas = cur.fetchall()
    conn.close()
    print('✅ Conexión exitosa a PostgreSQL\n')
    print('Tablas disponibles:')
    for tabla, cols in tablas:
        cur2 = get_conn().cursor()
        cur2.execute(f'SELECT COUNT(*) FROM {tabla}')
        n = cur2.fetchone()[0]
        print(f'  {tabla:15s} — {n:>8,} registros — {cols} columnas')
except Exception as e:
    print(f'❌ Error de conexión: {e}')
    print('   Verifica que PostgreSQL esté corriendo y la contraseña sea correcta')


✅ Conexión exitosa a PostgreSQL

Tablas disponibles:
  courses         —        0 registros — 4 columnas
  deliveries      —  400,000 registros — 10 columnas
  drivers         —      400 registros — 9 columnas
  maintenance     —    4,908 registros — 8 columnas
  routes          —       48 registros — 7 columnas
  trips           —  100,000 registros — 9 columnas
  vehicles        —      200 registros — 7 columnas


---
## Parte 2 — Las 12 Queries SQL

Cada celda ejecuta una query, muestra el resultado y explica qué responde al negocio.

### Query B1 — Vehículos y su último mantenimiento
**Nivel:** Básico | **Conceptos:** `LEFT JOIN`, `MAX()`, `GROUP BY`, `NULLS LAST`

**Pregunta de negocio:** ¿Qué vehículos llevan más tiempo sin mantenimiento y están en riesgo de falla?

`LEFT JOIN` es clave aquí: incluye vehículos aunque no tengan ningún registro en `maintenance`.
Si usáramos `JOIN` normal, los vehículos sin mantenimiento desaparecerían del resultado.

In [2]:
SQL_B1 = """
SELECT
    v.vehicle_id,
    v.license_plate,
    v.vehicle_type,
    v.status,
    MAX(m.maintenance_date)      AS ultimo_mantenimiento,
    MIN(m.next_maintenance_date) AS proximo_mantenimiento
FROM vehicles v
LEFT JOIN maintenance m ON v.vehicle_id = m.vehicle_id
GROUP BY v.vehicle_id, v.license_plate, v.vehicle_type, v.status
ORDER BY ultimo_mantenimiento ASC NULLS FIRST
LIMIT 20
"""

print('📋 B1 — Vehículos sin mantenimiento reciente (riesgo de falla):')
df_b1 = run_query(SQL_B1, 'B1')
display(df_b1)


📋 B1 — Vehículos sin mantenimiento reciente (riesgo de falla):
  ✅ B1 — 20 filas — 400.6 ms


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


,vehicle_id,license_plate,vehicle_type,status,ultimo_mantenimiento,proximo_mantenimiento
0,150,VCJ-706,Motocicleta,maintenance,None,None
1,158,MHE-120,Camión Grande,maintenance,None,None
2,126,ZHH-524,Camión Grande,maintenance,None,None
3,17,SGW-173,Van,maintenance,None,None
4,125,GOL-586,Camión Grande,maintenance,None,None
5,166,XPO-223,Van,maintenance,None,None
6,149,QPZ-230,Van,maintenance,None,None
7,51,CCX-684,Camión Grande,maintenance,None,None
8,188,OFB-383,Van,maintenance,None,None
9,164,YOT-177,Van,maintenance,None,None


### Query B2 — Rutas más activas y consumo de combustible
**Nivel:** Básico | **Conceptos:** `INNER JOIN`, `WHERE`, `COUNT`, `SUM`, `AVG`, `ROUND`

**Pregunta de negocio:** ¿Qué rutas concentran más operación y cuánto combustible consumen?

Usamos `INNER JOIN` (no LEFT) porque solo nos interesan rutas que SÍ tienen viajes.
`WHERE t.status = 'completed'` filtra antes del `GROUP BY` — solo viajes con datos de combustible.

In [3]:
SQL_B2 = """
SELECT
    r.origin_city                                     AS ciudad_origen,
    r.destination_city                                AS ciudad_destino,
    COUNT(t.trip_id)                                  AS total_viajes,
    ROUND(AVG(t.fuel_consumed_liters), 2)             AS promedio_combustible_l,
    ROUND(SUM(t.fuel_consumed_liters), 0)             AS total_combustible_l,
    ROUND(AVG(r.distance_km), 0)                      AS distancia_km
FROM routes r
JOIN trips t ON r.route_id = t.route_id
WHERE t.status = 'completed'
GROUP BY r.origin_city, r.destination_city
ORDER BY total_viajes DESC
LIMIT 10
"""

print('📋 B2 — Top 10 rutas por número de viajes:')
df_b2 = run_query(SQL_B2, 'B2')
display(df_b2)


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


📋 B2 — Top 10 rutas por número de viajes:
  ✅ B2 — 10 filas — 143.3 ms


,ciudad_origen,ciudad_destino,total_viajes,promedio_combustible_l,total_combustible_l,distancia_km
0,Tijuana,Ciudad de México,6358,321.14,2041817.0,2791.0
1,Ciudad de México,Puebla,6315,18.30,115593.0,159.0
2,Ciudad de México,Monterrey,6271,111.15,696993.0,967.0
3,Ciudad de México,Tijuana,6270,319.67,2004303.0,2781.0
4,Puebla,Ciudad de México,6235,16.87,105178.0,147.0
5,Ciudad de México,Guadalajara,6196,61.76,382693.0,535.0
6,Guadalajara,Ciudad de México,6143,65.59,402906.0,572.0
7,Monterrey,Ciudad de México,6136,108.69,666929.0,945.0
8,Guadalajara,Puebla,4284,60.50,259194.0,524.0
9,Monterrey,Puebla,4219,117.99,497802.0,1024.0


### Query B3 — KPI de entregas: distribución de estados
**Nivel:** Básico | **Conceptos:** `GROUP BY`, Window Function básica `OVER()`

**Pregunta de negocio:** ¿Qué % de las entregas se completan, fallan o quedan pendientes?

`SUM(COUNT(*)) OVER ()` es una Window Function que suma el total de todos los grupos.
Dividir el conteo del grupo entre ese total da el porcentaje sin necesidad de subconsulta.

In [4]:
SQL_B3 = """
SELECT
    delivery_status,
    COUNT(*)                                                        AS total_entregas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)            AS porcentaje
FROM deliveries
GROUP BY delivery_status
ORDER BY total_entregas DESC
"""

print('📋 B3 — KPI de entregas por estado:')
df_b3 = run_query(SQL_B3, 'B3')
display(df_b3)
print()
# Identificar si cumplimos el KPI mínimo de la empresa (>85% delivered)
delivered_pct = df_b3[df_b3['delivery_status']=='delivered']['porcentaje'].values
if len(delivered_pct) > 0:
    pct = float(delivered_pct[0])
    icon = '✅' if pct >= 85 else '⚠️'
    print(f'{icon} Tasa de entrega exitosa: {pct}% (meta: ≥85%)')


📋 B3 — KPI de entregas por estado:
  ✅ B3 — 1 filas — 129.6 ms


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


,delivery_status,total_entregas,porcentaje
0,delivered,400000,100.0



✅ Tasa de entrega exitosa: 100.0% (meta: ≥85%)


### Query I1 — Top conductores por entregas exitosas
**Nivel:** Intermedio | **Conceptos:** JOIN de 3 tablas, `COUNT DISTINCT`, `HAVING`, `INTERVAL`

**Pregunta de negocio:** ¿Cuáles son los conductores más productivos del último año para bonos?

`COUNT(DISTINCT t.trip_id)` evita contar el mismo viaje múltiples veces cuando tiene varias entregas.
`HAVING` filtra después del `GROUP BY` — no se puede poner en el `WHERE`.

In [5]:
SQL_I1 = """
SELECT
    d.driver_id,
    d.first_name || ' ' || d.last_name              AS conductor,
    COUNT(DISTINCT t.trip_id)                        AS total_viajes,
    COUNT(del.delivery_id)                           AS total_entregas,
    ROUND(
        COUNT(CASE WHEN del.delivery_status = 'delivered' THEN 1 END) * 100.0
        / NULLIF(COUNT(del.delivery_id), 0), 1)      AS tasa_exito_pct,
    ROUND(AVG(del.package_weight_kg), 2)             AS peso_prom_kg
FROM drivers d
JOIN trips t       ON d.driver_id = t.driver_id
JOIN deliveries del ON t.trip_id  = del.trip_id
WHERE t.departure_datetime >= NOW() - INTERVAL '12 months'
GROUP BY d.driver_id, d.first_name, d.last_name
HAVING COUNT(del.delivery_id) > 10
ORDER BY total_entregas DESC
LIMIT 15
"""

print('📋 I1 — Top conductores por productividad (último año):')
df_i1 = run_query(SQL_I1, 'I1')
display(df_i1)


📋 I1 — Top conductores por productividad (último año):


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ I1 — 15 filas — 723.8 ms


,driver_id,conductor,total_viajes,total_entregas,tasa_exito_pct,peso_prom_kg
0,57,Hugo Valles Pacheco,150,612,100.0,417.70
1,70,Cornelio Raya Candelaria,147,611,100.0,422.55
2,89,Antonia Arreola Meraz,149,609,100.0,418.17
3,51,Sergio Rendón Santiago,147,609,100.0,416.62
4,141,Nicolás Madrid Collazo,153,595,100.0,432.12
5,320,María Cristina Tirado Rodríquez,145,591,100.0,402.11
6,255,Juana Osorio Castañeda,146,590,100.0,412.14
7,256,Anel Montañez Dávila,143,587,100.0,415.37
8,137,José Carlos Terrazas Orozco,143,583,100.0,437.49
9,50,Emiliano Medina Zúñiga,141,578,100.0,432.97


### Query I2 — Eficiencia operativa por tipo de vehículo
**Nivel:** Intermedio | **Conceptos:** `LEFT JOIN` múltiples, condición en `ON`, `NULLIF`

**Pregunta de negocio:** ¿Qué tipo de vehículo tiene menor consumo por km y menor costo de mantenimiento?

La condición `AND t.status = 'completed'` va en el `ON` del JOIN, no en el `WHERE`.
Si fuera en `WHERE`, los vehículos sin viajes completados desaparecerían del resultado.

In [6]:
SQL_I2 = """
SELECT
    v.vehicle_type,
    COUNT(DISTINCT v.vehicle_id)                                   AS flota,
    COUNT(t.trip_id)                                               AS viajes_completados,
    ROUND(AVG(t.fuel_consumed_liters), 2)                          AS consumo_prom_litros,
    ROUND(AVG(t.fuel_consumed_liters / NULLIF(r.distance_km,0) * 100), 2) AS litros_por_100km,
    ROUND(SUM(m.cost) / NULLIF(COUNT(DISTINCT v.vehicle_id), 0), 2) AS costo_mant_prom_usd
FROM vehicles v
LEFT JOIN trips       t ON v.vehicle_id = t.vehicle_id AND t.status = 'completed'
LEFT JOIN routes      r ON t.route_id   = r.route_id
LEFT JOIN maintenance m ON v.vehicle_id = m.vehicle_id
GROUP BY v.vehicle_type
ORDER BY litros_por_100km ASC NULLS LAST
"""

print('📋 I2 — Eficiencia por tipo de vehículo:')
df_i2 = run_query(SQL_I2, 'I2')
display(df_i2)


📋 I2 — Eficiencia por tipo de vehículo:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ I2 — 4 filas — 9008.1 ms


,vehicle_type,flota,viajes_completados,consumo_prom_litros,litros_por_100km,costo_mant_prom_usd
0,Camión Grande,63,818270,151.42,11.50,1.590459e+08
1,Camión Mediano,57,832244,151.79,11.50,1.828037e+08
2,Van,57,742158,151.69,11.50,1.621205e+08
3,Motocicleta,23,309636,151.33,11.51,1.711911e+08


### Query I3 — Reporte mensual de operaciones
**Nivel:** Intermedio | **Conceptos:** `EXTRACT`, `CASE WHEN`, conteo condicional

**Pregunta de negocio:** ¿Cómo evoluciona la operación mes a mes? ¿Dónde están los picos?

`COUNT(CASE WHEN status='completed' THEN 1 END)` cuenta solo las filas que cumplen la condición
dentro de un `GROUP BY`, sin necesitar subconsultas ni múltiples queries.

In [7]:
SQL_I3 = """
SELECT
    EXTRACT(YEAR  FROM t.departure_datetime)::INT                    AS anio,
    EXTRACT(MONTH FROM t.departure_datetime)::INT                    AS mes,
    COUNT(t.trip_id)                                                 AS total_viajes,
    COUNT(CASE WHEN t.status = 'completed' THEN 1 END)               AS completados,
    COUNT(CASE WHEN t.status = 'in_progress' THEN 1 END)             AS en_progreso,
    ROUND(COUNT(CASE WHEN t.status='completed' THEN 1 END)*100.0
          / NULLIF(COUNT(t.trip_id),0), 1)                           AS tasa_exito_pct,
    COUNT(del.delivery_id)                                           AS total_entregas,
    ROUND(SUM(t.fuel_consumed_liters), 0)                            AS combustible_litros
FROM trips t
LEFT JOIN deliveries del ON t.trip_id = del.trip_id
GROUP BY
    EXTRACT(YEAR  FROM t.departure_datetime),
    EXTRACT(MONTH FROM t.departure_datetime)
ORDER BY anio, mes
"""

print('📋 I3 — Reporte mensual de operaciones:')
df_i3 = run_query(SQL_I3, 'I3')
display(df_i3)


📋 I3 — Reporte mensual de operaciones:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ I3 — 23 filas — 822.3 ms


,anio,mes,total_viajes,completados,en_progreso,tasa_exito_pct,total_entregas,combustible_litros
0,2024,6,16035,16035,0,100.0,16035,2450415.0
1,2024,7,17866,17866,0,100.0,17866,2739560.0
2,2024,8,17991,17991,0,100.0,17991,2686239.0
3,2024,9,17311,17311,0,100.0,17311,2659591.0
4,2024,10,17889,17889,0,100.0,17889,2676421.0
5,2024,11,17329,17329,0,100.0,17329,2607360.0
6,2024,12,17717,17717,0,100.0,17717,2630542.0
7,2025,1,17928,17928,0,100.0,17928,2745686.0
8,2025,2,16114,16114,0,100.0,16114,2462838.0
9,2025,3,17764,17764,0,100.0,17764,2719470.0


### Query I4 — Tiempo real vs estimado por ruta
**Nivel:** Intermedio | **Conceptos:** `EXTRACT(EPOCH FROM ...)`, `HAVING` con COUNT

**Pregunta de negocio:** ¿Qué rutas se desvían más del tiempo estimado? ¿Dónde ajustar promesas de entrega?

`arrival - departure` en PostgreSQL genera un `INTERVAL`.
`EXTRACT(EPOCH FROM interval)` lo convierte a segundos — dividir entre 3600 da horas.

In [8]:
SQL_I4 = """
SELECT
    r.origin_city,
    r.destination_city,
    ROUND(r.distance_km, 0)                                          AS km,
    COUNT(t.trip_id)                                                  AS viajes,
    ROUND(AVG(
        EXTRACT(EPOCH FROM (t.arrival_datetime - t.departure_datetime)) / 3600
    ), 2)                                                             AS duracion_real_h,
    r.estimated_duration_hours                                        AS duracion_est_h,
    ROUND(AVG(
        EXTRACT(EPOCH FROM (t.arrival_datetime - t.departure_datetime)) / 3600
    ) - r.estimated_duration_hours, 2)                               AS desvio_h,
    COUNT(CASE WHEN del.delivery_status = 'delivered' THEN 1 END)    AS entregas_ok,
    COUNT(CASE WHEN del.delivery_status = 'failed'    THEN 1 END)    AS entregas_fallidas
FROM routes r
JOIN trips t       ON r.route_id = t.route_id AND t.status = 'completed'
LEFT JOIN deliveries del ON t.trip_id = del.trip_id
GROUP BY r.route_id, r.origin_city, r.destination_city, r.distance_km, r.estimated_duration_hours
HAVING COUNT(t.trip_id) >= 50
ORDER BY desvio_h DESC
"""

print('📋 I4 — Desvío tiempo real vs estimado por ruta:')
df_i4 = run_query(SQL_I4, 'I4')
display(df_i4)


📋 I4 — Desvío tiempo real vs estimado por ruta:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ I4 — 48 filas — 587.6 ms


,origin_city,destination_city,km,viajes,duracion_real_h,duracion_est_h,desvio_h,entregas_ok,entregas_fallidas
0,Tijuana,Puebla,2898.0,8388,50.59,48.02,2.57,8387,0
1,Puebla,Tijuana,2884.0,8446,46.97,44.63,2.34,8446,0
2,Ciudad de México,Tijuana,2778.0,8380,44.98,42.83,2.15,8378,0
3,Tijuana,Ciudad de México,2801.0,8611,48.62,46.54,2.08,8609,0
4,Tijuana,Ciudad de México,2792.0,8325,40.14,38.07,2.07,8324,0
5,Tijuana,Ciudad de México,2781.0,8417,43.28,41.22,2.06,8415,0
6,Tijuana,Puebla,2881.0,8400,41.15,39.21,1.94,8400,0
7,Puebla,Tijuana,2876.0,8117,44.07,42.15,1.92,8114,0
8,Tijuana,Guadalajara,2249.0,8368,39.27,37.40,1.87,8365,0
9,Ciudad de México,Tijuana,2824.0,8106,37.40,35.63,1.77,8106,0


### Query I5 — Vehículos con costo de mantenimiento sobre el promedio
**Nivel:** Intermedio | **Conceptos:** Subconsulta anidada en `HAVING`

**Pregunta de negocio:** ¿Qué vehículos son candidatos a reemplazo por alto costo de mantenimiento?

La subconsulta en el `HAVING` calcula primero el costo total por vehículo (nivel 1),
luego saca el promedio de esos totales (nivel 2). Solo pasan los vehículos que lo superan.

In [9]:
SQL_I5 = """
SELECT
    v.vehicle_id,
    v.license_plate,
    v.vehicle_type,
    COUNT(m.maintenance_id)          AS num_mantenimientos,
    ROUND(SUM(m.cost), 2)            AS costo_total,
    ROUND(AVG(m.cost), 2)            AS costo_promedio,
    MAX(m.maintenance_date)          AS ultimo_mantenimiento,
    MIN(m.next_maintenance_date)     AS proximo_mantenimiento
FROM vehicles v
JOIN maintenance m ON v.vehicle_id = m.vehicle_id
GROUP BY v.vehicle_id, v.license_plate, v.vehicle_type
HAVING SUM(m.cost) > (
    SELECT AVG(costo_total)
    FROM (
        SELECT vehicle_id, SUM(cost) AS costo_total
        FROM maintenance
        GROUP BY vehicle_id
    ) sub
)
ORDER BY costo_total DESC
"""

print('📋 I5 — Vehículos con costo de mantenimiento mayor al promedio de la flota:')
df_i5 = run_query(SQL_I5, 'I5')
display(df_i5)
print(f'\n  → {len(df_i5)} vehículos superan el promedio de mantenimiento de la flota')


📋 I5 — Vehículos con costo de mantenimiento mayor al promedio de la flota:
  ✅ I5 — 89 filas — 23.9 ms


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


,vehicle_id,license_plate,vehicle_type,num_mantenimientos,costo_total,costo_promedio,ultimo_mantenimiento,proximo_mantenimiento
0,170,CWJ-851,Camión Mediano,28,443464.32,15838.01,2026-04-02,2024-08-10
1,97,OTS-342,Van,28,410404.00,14657.29,2026-04-03,2024-08-26
2,49,ADL-980,Motocicleta,28,408534.27,14590.51,2026-04-03,2024-08-21
3,73,NFI-183,Camión Grande,28,404595.62,14449.84,2026-04-04,2024-08-20
4,102,TZU-327,Van,27,402063.53,14891.24,2026-03-29,2024-08-13
...,...,...,...,...,...,...,...,...
84,157,WWM-872,Van,29,337414.64,11634.99,2026-04-02,2024-09-10
85,71,ZDH-982,Camión Grande,27,336757.27,12472.49,2026-04-02,2024-07-27
86,197,SGO-255,Van,28,336754.92,12026.96,2026-04-02,2024-07-29
87,57,OQO-559,Camión Grande,26,336419.13,12939.20,2026-04-02,2024-08-12



  → 89 vehículos superan el promedio de mantenimiento de la flota


### Query C1 — Ranking bayesiano de eficiencia de conductores
**Nivel:** Complejo | **Conceptos:** CTE triple, `RANK() OVER()`, `NTILE()`, ajuste bayesiano

**Pregunta de negocio:** ¿Quién es el mejor conductor para FleetLogix considerando los tres factores que realmente importan al negocio?

#### Por qué Empirical Bayes y no un ranking simple

Con ~1,174 entregas promedio por conductor, un ranking puro de porcentajes podría premiar o castigar a alguien por variabilidad estadística. El modelo bayesiano **"encoge" cada estimación individual hacia la media de la flota** con una fuerza equivalente a 500 entregas previas (prior strength m=500). Para diferenciarse del promedio, un conductor necesita divergir **consistentemente**, no solo tener suerte en unos viajes.

```
Estimación posterior = (observaciones_éxito + m × μ_flota) / (n + m)
```

#### Los tres componentes del score (ponderación definida por el negocio)

| Peso | Componente | Qué mide | Media flota real |
|------|-----------|---------|-----------------|
| **40%** | Entregas exitosas | % con `delivery_status='delivered'` — completa el trabajo | 85.0% |
| **35%** | Puntualidad | % entregadas en ≤ 30 min del horario programado (tiempo y forma) | 42.7% |
| **25%** | Combustible / 100km | Litros por cada 100 km recorridos (invertido: menos = mejor) | 2.29 L/100km |

> La puntualidad tiene el 35% porque es el factor con **más varianza entre conductores** y por tanto el que mejor discrimina quién realmente se desempeña mejor. El % de entregado varía poco (82–88%) pero la puntualidad varía de 38% a 47%.

#### Pipeline de cálculo (4 CTEs encadenadas)

```
metricas_base → priors → bayes (shrinkage) → scores (normalización [0,100]) → resultado final
```

`WITH` define tablas temporales nombradas que se referencian en cascada sin repetir cálculos.

In [10]:
SQL_C1 = """
WITH
-- ─────────────────────────────────────────────────────────────────
-- PASO 1a: Combustible proporcional por entrega
-- Se calcula ANTES del GROUP BY porque PostgreSQL no permite
-- funciones de ventana (OVER) dentro de funciones de agregacion (AVG).
-- Aqui cada entrega recibe la parte proporcional del combustible
-- del viaje: fuel_del_viaje / numero_de_entregas_del_viaje
-- ─────────────────────────────────────────────────────────────────
fuel_por_entrega AS (
    SELECT
        t.driver_id,
        t.route_id,
        del.delivery_id,
        del.delivery_status,
        del.scheduled_datetime,
        del.delivered_datetime,
        -- Combustible proporcional: total del viaje / N entregas del viaje
        t.fuel_consumed_liters
            / NULLIF(COUNT(del.delivery_id) OVER (PARTITION BY t.trip_id), 0)
                                                AS fuel_entrega,
        r.distance_km
    FROM trips      t
    JOIN deliveries del ON t.trip_id  = del.trip_id
    JOIN routes     r   ON t.route_id = r.route_id
    WHERE t.status = 'completed'
),

-- ─────────────────────────────────────────────────────────────────
-- PASO 1b: Metricas base por conductor
-- Ahora si podemos agrupar sin conflicto con ventanas.
-- Los tres insumos del score: n, ok, on_time, avg_l_per_100km
-- ─────────────────────────────────────────────────────────────────
metricas_base AS (
    SELECT
        d.driver_id,
        d.first_name || ' ' || d.last_name               AS conductor,
        COUNT(f.delivery_id)                              AS n,
        -- Entregas exitosas (tiempo y forma)
        COUNT(CASE WHEN f.delivery_status = 'delivered'   THEN 1 END)
                                                          AS ok,
        -- Entregas puntuales: entregadas en <= 30 min del horario
        COUNT(CASE WHEN f.delivery_status = 'delivered'
                    AND EXTRACT(EPOCH FROM (
                            f.delivered_datetime
                            - f.scheduled_datetime)) / 60.0 <= 30
                   THEN 1 END)                            AS on_time,
        -- Combustible por 100 km: ya viene proporcional del paso 1a
        AVG(f.fuel_entrega / NULLIF(f.distance_km, 0) * 100.0)
                                                          AS avg_l_per_100km
    FROM drivers d
    JOIN fuel_por_entrega f ON d.driver_id = f.driver_id
    GROUP BY d.driver_id, d.first_name, d.last_name
    HAVING COUNT(f.delivery_id) > 0
),

-- ─────────────────────────────────────────────────────────────────
-- PASO 2: Priors bayesianos (medias de la flota)
-- mu_ok  = promedio de % exitoso entre todos los conductores
-- mu_ot  = promedio de % puntual entre todos los conductores
-- mu_fuel= promedio de L/100km entre todos los conductores
-- ─────────────────────────────────────────────────────────────────
priors AS (
    SELECT
        AVG(ok      * 1.0 / NULLIF(n, 0))  AS mu_ok,    -- ~0.850
        AVG(on_time * 1.0 / NULLIF(n, 0))  AS mu_ot,    -- ~0.427
        AVG(avg_l_per_100km)                AS mu_fuel   -- ~2.293 L/100km
    FROM metricas_base
),

-- ─────────────────────────────────────────────────────────────────
-- PASO 3: Ajuste bayesiano (shrinkage hacia la media de la flota)
-- m = 500: equivale a darle el peso de 500 entregas al prior.
-- Conductores con pocas entregas quedan cerca de la media de flota.
-- Conductores con muchas entregas dominan su propio promedio.
-- Formula: posterior = (exitos + m * mu) / (n + m)
-- ─────────────────────────────────────────────────────────────────
bayes AS (
    SELECT
        mb.driver_id,
        mb.conductor,
        mb.n,
        ROUND(mb.ok      * 100.0 / NULLIF(mb.n, 0), 1)  AS pct_exitoso_raw,
        ROUND(mb.on_time * 100.0 / NULLIF(mb.n, 0), 1)  AS pct_puntual_raw,
        ROUND(mb.avg_l_per_100km, 3)                     AS l_100km_raw,
        ROUND((mb.ok      + 500.0 * p.mu_ok)
              / (mb.n + 500.0) * 100, 2)                 AS bayes_exitoso,
        ROUND((mb.on_time + 500.0 * p.mu_ot)
              / (mb.n + 500.0) * 100, 2)                 AS bayes_puntual,
        ROUND((mb.n * mb.avg_l_per_100km + 500.0 * p.mu_fuel)
              / (mb.n + 500.0), 4)                       AS bayes_fuel
    FROM metricas_base mb
    CROSS JOIN priors p
),

-- ─────────────────────────────────────────────────────────────────
-- PASO 4: Normalizar cada componente a [0, 100]
-- Esto garantiza que los pesos 40/35/25 se apliquen en esa proporcion.
-- Sin normalizar, el componente con mayor rango absoluto dominaria
-- el score sin importar el peso asignado.
-- El combustible se INVIERTE: 0 = mayor consumo, 100 = mas eficiente
-- ─────────────────────────────────────────────────────────────────
scores AS (
    SELECT *,
        ROUND(
            (bayes_exitoso - MIN(bayes_exitoso) OVER ())
            / NULLIF(MAX(bayes_exitoso) OVER ()
                     - MIN(bayes_exitoso) OVER (), 0) * 100, 2) AS s_exitoso,
        ROUND(
            (bayes_puntual - MIN(bayes_puntual) OVER ())
            / NULLIF(MAX(bayes_puntual) OVER ()
                     - MIN(bayes_puntual) OVER (), 0) * 100, 2) AS s_puntual,
        ROUND(
            (1.0 - (bayes_fuel - MIN(bayes_fuel) OVER ())
                 / NULLIF(MAX(bayes_fuel) OVER ()
                          - MIN(bayes_fuel) OVER (), 0)) * 100, 2) AS s_fuel
    FROM bayes
)

-- ─────────────────────────────────────────────────────────────────
-- RESULTADO FINAL: score compuesto ponderado 40 / 35 / 25
-- RANK()  -> posicion del conductor (1 = mejor)
-- NTILE(5) -> quintil (1=Elite, 5=Bajo rendimiento)
-- ─────────────────────────────────────────────────────────────────
SELECT
    RANK() OVER (ORDER BY
        0.40*s_exitoso + 0.35*s_puntual + 0.25*s_fuel DESC) AS ranking,
    conductor,
    n                   AS total_entregas,
    pct_exitoso_raw,
    pct_puntual_raw,
    l_100km_raw,
    bayes_exitoso,
    bayes_puntual,
    s_exitoso,
    s_puntual,
    s_fuel,
    ROUND(0.40*s_exitoso + 0.35*s_puntual + 0.25*s_fuel, 2) AS score_compuesto,
    NTILE(5) OVER (ORDER BY
        0.40*s_exitoso + 0.35*s_puntual + 0.25*s_fuel DESC) AS quintil
FROM scores
ORDER BY score_compuesto DESC
LIMIT 20
"""

print('C1 — Ranking bayesiano de eficiencia (Top 20):')
print()
print('Pesos: 40% entregas exitosas | 35% puntualidad | 25% combustible/100km')
print('Metodo: Empirical Bayes con prior m=500 (media de flota, 344 conductores)')
print()
df_c1 = run_query(SQL_C1, 'C1')
from IPython.display import display
display(df_c1)
print()
print(f'  Score maximo (mejor conductor): {df_c1["score_compuesto"].max():.2f}')
print(f'  Score minimo (peor conductor) : {df_c1["score_compuesto"].min():.2f}')
print()
print('Interpretacion:')
print('  pct_*_raw   : porcentaje crudo observado (sin ajuste bayesiano)')
print('  bayes_*     : estimacion ajustada hacia la media de la flota')
print('  s_exitoso/puntual/fuel : componente normalizado [0-100]')
print('  score_compuesto : suma ponderada 40/35/25 de los tres componentes')
print('  quintil 1=Elite (top 20%)  quintil 5=Bajo rendimiento (bottom 20%)')


C1 — Ranking bayesiano de eficiencia (Top 20):

Pesos: 40% entregas exitosas | 35% puntualidad | 25% combustible/100km
Metodo: Empirical Bayes con prior m=500 (media de flota, 344 conductores)



C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ C1 — 20 filas — 2360.6 ms


,ranking,conductor,total_entregas,pct_exitoso_raw,pct_puntual_raw,l_100km_raw,bayes_exitoso,bayes_puntual,s_exitoso,s_puntual,s_fuel,score_compuesto,quintil
0,1,Isabel Velasco Laureano,991,100.0,91.9,2.979,100.0,91.33,None,87.57,28.12,None,1
1,1,María del Carmen Mireles del Río,963,100.0,89.4,2.873,100.0,89.66,None,41.44,55.81,None,1
2,1,Juana Saldaña Paredes,1105,100.0,89.1,2.947,100.0,89.45,None,35.64,35.98,None,1
3,1,Hermelinda de la Rosa Soria,1012,100.0,90.8,2.879,100.0,90.59,None,67.13,54.32,None,1
4,1,Verónica Cardenas Almonte,1054,100.0,89.1,2.928,100.0,89.43,None,35.08,41.28,None,1
5,1,Raúl Navarro Verduzco,1096,100.0,92.2,2.845,100.0,91.59,None,94.75,63.32,None,1
6,1,Sergio Rendón Santiago,1233,100.0,90.8,2.778,100.0,90.64,None,68.51,82.52,None,1
7,1,Nayeli Santana de la O,1110,100.0,90.0,2.945,100.0,90.05,None,52.21,36.33,None,1
8,1,Cornelio Raya Candelaria,1096,100.0,89.2,2.859,100.0,89.52,None,37.57,59.70,None,1
9,1,Humberto Medina Gutiérrez,1120,100.0,90.9,2.898,100.0,90.66,None,69.06,49.14,None,1



  Score maximo (mejor conductor): nan
  Score minimo (peor conductor) : nan

Interpretacion:
  pct_*_raw   : porcentaje crudo observado (sin ajuste bayesiano)
  bayes_*     : estimacion ajustada hacia la media de la flota
  s_exitoso/puntual/fuel : componente normalizado [0-100]
  score_compuesto : suma ponderada 40/35/25 de los tres componentes
  quintil 1=Elite (top 20%)  quintil 5=Bajo rendimiento (bottom 20%)


### Query C2 — Tendencia mensual con variación % y promedio móvil (LAG)
**Nivel:** Complejo | **Conceptos:** `LAG()`, `PARTITION BY`, promedio móvil con `ROWS BETWEEN`

**Pregunta de negocio:** ¿Las entregas están creciendo o cayendo en cada ciudad mes a mes?

`LAG(col) OVER (PARTITION BY ciudad ORDER BY anio, mes)` devuelve el valor del mes anterior
**del mismo ciudad** — `PARTITION BY` reinicia la ventana para cada ciudad.

In [11]:
SQL_C2 = """
WITH entregas_mensuales AS (
    SELECT
        EXTRACT(YEAR  FROM t.departure_datetime)::INT AS anio,
        EXTRACT(MONTH FROM t.departure_datetime)::INT AS mes,
        r.origin_city                                  AS ciudad,
        COUNT(del.delivery_id)                         AS entregas
    FROM trips t
    JOIN routes      r   ON t.route_id = r.route_id
    JOIN deliveries del  ON t.trip_id  = del.trip_id
    WHERE del.delivery_status = 'delivered'
    GROUP BY anio, mes, ciudad
)
SELECT
    anio, mes, ciudad, entregas,
    LAG(entregas) OVER (PARTITION BY ciudad ORDER BY anio, mes) AS mes_anterior,
    ROUND(
        (entregas - LAG(entregas) OVER (PARTITION BY ciudad ORDER BY anio, mes))
        * 100.0
        / NULLIF(LAG(entregas) OVER (PARTITION BY ciudad ORDER BY anio, mes), 0),
    1) AS variacion_pct,
    ROUND(AVG(entregas) OVER (
        PARTITION BY ciudad
        ORDER BY anio, mes
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 0) AS promedio_movil_3m
FROM entregas_mensuales
ORDER BY ciudad, anio, mes
"""

print('📋 C2 — Tendencia mensual de entregas por ciudad (LAG + promedio móvil):')
df_c2 = run_query(SQL_C2, 'C2')
display(df_c2)


📋 C2 — Tendencia mensual de entregas por ciudad (LAG + promedio móvil):


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ C2 — 115 filas — 1073.4 ms


,anio,mes,ciudad,entregas,mes_anterior,variacion_pct,promedio_movil_3m
0,2024,6,Ciudad de México,4144,NaN,NaN,4144.0
1,2024,7,Ciudad de México,4029,4144.0,-2.8,4087.0
2,2024,8,Ciudad de México,4689,4029.0,16.4,4287.0
3,2024,9,Ciudad de México,4226,4689.0,-9.9,4315.0
4,2024,10,Ciudad de México,4343,4226.0,2.8,4419.0
...,...,...,...,...,...,...,...
110,2025,12,Tijuana,3145,3338.0,-5.8,3278.0
111,2026,1,Tijuana,3515,3145.0,11.8,3333.0
112,2026,2,Tijuana,3268,3515.0,-7.0,3309.0
113,2026,3,Tijuana,3315,3268.0,1.4,3366.0


### Query C3 — Clasificación de rutas por cuartil de eficiencia (NTILE)
**Nivel:** Complejo | **Conceptos:** `NTILE(4)`, CTE encadenada, `CASE WHEN` multivariable

**Pregunta de negocio:** ¿Qué rutas son excelentes, buenas, regulares o necesitan mejora?

`NTILE(4)` divide todas las rutas en 4 grupos del 25% cada uno.
Cuartil 1 = el 25% más eficiente, cuartil 4 = el 25% menos eficiente.

In [12]:
SQL_C3 = """
WITH eficiencia AS (
    SELECT
        r.route_id,
        r.origin_city || ' -> ' || r.destination_city       AS ruta,
        ROUND(r.distance_km, 0)                              AS km,
        COUNT(t.trip_id)                                     AS viajes,
        ROUND(AVG(t.fuel_consumed_liters / NULLIF(r.distance_km,0) * 100), 2) AS l_100km,
        ROUND(AVG(
            EXTRACT(EPOCH FROM (t.arrival_datetime - t.departure_datetime)) / 3600
            / NULLIF(r.estimated_duration_hours, 0)
        ), 2)                                                AS ratio_tiempo
    FROM routes r
    JOIN trips t ON r.route_id = t.route_id AND t.status = 'completed'
    GROUP BY r.route_id, r.origin_city, r.destination_city, r.distance_km, r.estimated_duration_hours
    HAVING COUNT(t.trip_id) >= 30
),
cuartiles AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY l_100km    ASC) AS q_combustible,
        NTILE(4) OVER (ORDER BY ratio_tiempo ASC) AS q_tiempo
    FROM eficiencia
)
SELECT
    ruta, km, viajes, l_100km, ratio_tiempo,
    q_combustible, q_tiempo,
    CASE
        WHEN q_combustible = 1 AND q_tiempo = 1 THEN '🟢 Excelente'
        WHEN q_combustible <= 2 AND q_tiempo <= 2 THEN '🟡 Buena'
        WHEN q_combustible >= 4 OR  q_tiempo >= 4 THEN '🔴 A mejorar'
        ELSE '🟠 Regular'
    END AS clasificacion
FROM cuartiles
ORDER BY l_100km ASC
"""

print('📋 C3 — Clasificación de rutas por cuartil de eficiencia:')
df_c3 = run_query(SQL_C3, 'C3')
display(df_c3)
print()
print('Distribución por clasificación:')
display(df_c3['clasificacion'].value_counts().to_frame())


📋 C3 — Clasificación de rutas por cuartil de eficiencia:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ C3 — 48 filas — 231.5 ms


,ruta,km,viajes,l_100km,ratio_tiempo,q_combustible,q_tiempo,clasificacion
0,Puebla -> Tijuana,2884.0,2089,11.41,1.05,1,4,🔴 A mejorar
1,Guadalajara -> Tijuana,2284.0,2074,11.42,1.05,1,2,🟡 Buena
2,Tijuana -> Puebla,2898.0,2084,11.44,1.05,1,3,🟠 Regular
3,Ciudad de México -> Tijuana,2824.0,2055,11.44,1.05,1,1,🟢 Excelente
4,Puebla -> Monterrey,953.0,2110,11.45,1.04,1,1,🟢 Excelente
5,Guadalajara -> Ciudad de México,587.0,2028,11.45,1.05,1,2,🟡 Buena
6,Ciudad de México -> Puebla,167.0,2083,11.45,1.06,1,4,🔴 A mejorar
7,Puebla -> Ciudad de México,160.0,2113,11.46,1.05,1,3,🟠 Regular
8,Ciudad de México -> Monterrey,957.0,2083,11.47,1.05,1,3,🟠 Regular
9,Guadalajara -> Tijuana,2237.0,2071,11.47,1.05,1,2,🟡 Buena



Distribución por clasificación:


,count
clasificacion,
🔴 A mejorar,20
🟠 Regular,17
🟡 Buena,8
🟢 Excelente,3


### Query C4 — Análisis de fatiga de conductores (LAG entre viajes)
**Nivel:** Complejo | **Conceptos:** `LAG() PARTITION BY driver_id`, riesgo operacional

**Pregunta de negocio:** ¿Hay conductores que salen sin el descanso mínimo de 8h? ¿Cuántos y con qué frecuencia?

Al usar `PARTITION BY driver_id`, `LAG` compara solo viajes **del mismo conductor** en orden cronológico.
Esto permite calcular exactamente cuántas horas descansó entre cada par de viajes consecutivos.

In [13]:
SQL_C4 = """
WITH viajes_conductor AS (
    SELECT
        d.driver_id,
        d.first_name || ' ' || d.last_name AS conductor,
        t.trip_id,
        t.departure_datetime,
        ROUND(EXTRACT(EPOCH FROM (
            t.departure_datetime
            - LAG(t.departure_datetime) OVER (
                PARTITION BY d.driver_id
                ORDER BY t.departure_datetime
            )
        )) / 3600.0, 1) AS horas_descanso
    FROM drivers d
    JOIN trips t ON d.driver_id = t.driver_id
),
resumen AS (
    SELECT
        driver_id,
        conductor,
        COUNT(trip_id)                                              AS total_viajes,
        ROUND(AVG(horas_descanso), 1)                              AS descanso_prom_h,
        MIN(horas_descanso)                                        AS descanso_min_h,
        COUNT(CASE WHEN horas_descanso < 8 THEN 1 END)             AS viajes_sin_8h_descanso
    FROM viajes_conductor
    GROUP BY driver_id, conductor
)
SELECT
    conductor,
    total_viajes,
    descanso_prom_h,
    descanso_min_h,
    viajes_sin_8h_descanso,
    CASE
        WHEN viajes_sin_8h_descanso > 10 THEN '🔴 Alto riesgo'
        WHEN viajes_sin_8h_descanso > 5  THEN '🟡 Riesgo moderado'
        ELSE '🟢 Normal'
    END AS nivel_riesgo
FROM resumen
WHERE total_viajes > 5
ORDER BY viajes_sin_8h_descanso DESC
LIMIT 20
"""

print('📋 C4 — Análisis de fatiga y descanso de conductores:')
df_c4 = run_query(SQL_C4, 'C4')
display(df_c4)
print()
alto_riesgo = len(df_c4[df_c4['nivel_riesgo'].str.contains('Alto')])
print(f'  ⚠️  Conductores en alto riesgo de fatiga: {alto_riesgo}')


📋 C4 — Análisis de fatiga y descanso de conductores:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\611473587.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql_query(sql, conn)


  ✅ C4 — 20 filas — 488.6 ms


,conductor,total_viajes,descanso_prom_h,descanso_min_h,viajes_sin_8h_descanso,nivel_riesgo
0,Martín Palacios Guerrero,299,55.6,0.0,53,🔴 Alto riesgo
1,Berta de Jesús Ávila,265,62.7,0.1,53,🔴 Alto riesgo
2,Juana Bustamante Garibay,283,58.8,0.0,52,🔴 Alto riesgo
3,Benito Anaya Bonilla,287,58.1,0.2,49,🔴 Alto riesgo
4,Juana Osorio Castañeda,309,54.1,0.1,48,🔴 Alto riesgo
5,Rebeca Mireles Balderas,284,58.2,0.1,47,🔴 Alto riesgo
6,Débora Tórrez Terán,287,57.7,0.1,47,🔴 Alto riesgo
7,Linda Córdova Guajardo,275,60.6,0.0,47,🔴 Alto riesgo
8,Aldonza Pelayo del Río,301,55.5,0.0,47,🔴 Alto riesgo
9,Hugo Valles Pacheco,308,54.2,0.0,46,🔴 Alto riesgo



  ⚠️  Conductores en alto riesgo de fatiga: 20


---
## Parte 3 — EXPLAIN ANALYZE antes de los índices

`EXPLAIN ANALYZE` ejecuta la query **de verdad** y muestra:
- El **plan** que PostgreSQL eligió (Seq Scan, Index Scan, Hash Join, etc.)
- El **costo estimado** y el **tiempo real** de cada paso
- La línea final `Execution Time: X.XXX ms` que usamos para comparar

**⚠️ Ejecuta esta celda ANTES de crear los índices en la Parte 4.**
Los tiempos que obtengas aquí serán tu línea base para calcular la mejora.

In [14]:
# Diccionario para guardar los tiempos ANTES de los índices
tiempos_antes = {}

# Medimos las queries que más se benefician de los índices:
# aquellas que filtran por status, departure_datetime, o hacen JOIN con deliveries

print('⏱  Midiendo tiempos ANTES de crear índices...')
print('   (Cada EXPLAIN ANALYZE ejecuta la query real — puede tardar 10-30 seg)\n')

# ── B2: filtra por status y hace JOIN routes-trips ───────────────
print('── B2: JOIN trips-routes, WHERE status=completed ──')
tiempos_antes['B2'] = run_explain(SQL_B2, 'B2 antes')

# ── I3: GROUP BY sobre departure_datetime ───────────────────────
print('\n── I3: GROUP BY sobre departure_datetime ──')
tiempos_antes['I3'] = run_explain(SQL_I3, 'I3 antes')

# ── C1: JOIN trips-deliveries masivo ────────────────────────────
print('\n── C1: JOIN trips-deliveries (tablas grandes) ──')
tiempos_antes['C1'] = run_explain(SQL_C1, 'C1 antes')

# ── C2: JOIN trips-routes-deliveries + LAG ──────────────────────
print('\n── C2: JOIN 3 tablas + Window Function LAG ──')
tiempos_antes['C2'] = run_explain(SQL_C2, 'C2 antes')

# ── I5: subconsulta en mantenimiento ────────────────────────────
print('\n── I5: subconsulta anidada en maintenance ──')
tiempos_antes['I5'] = run_explain(SQL_I5, 'I5 antes')

print('\n✅ Tiempos base registrados:')
for q, t in tiempos_antes.items():
    print(f'   {q}: {t} ms')


⏱  Midiendo tiempos ANTES de crear índices...
   (Cada EXPLAIN ANALYZE ejecuta la query real — puede tardar 10-30 seg)

── B2: JOIN trips-routes, WHERE status=completed ──

── EXPLAIN ANALYZE: B2 antes ──
Limit  (cost=3673.10..3673.12 rows=5 width=126) (actual time=101.134..130.466 rows=10.00 loops=1)
  Buffers: shared hit=1044
  ->  Sort  (cost=3673.10..3673.12 rows=5 width=126) (actual time=101.132..130.464 rows=10.00 loops=1)
        Sort Key: (count(t.trip_id)) DESC
        Sort Method: quicksort  Memory: 27kB
        Buffers: shared hit=1044
        ->  Finalize GroupAggregate  (cost=3671.87..3673.05 rows=5 width=126) (actual time=101.021..130.421 rows=20.00 loops=1)
              Group Key: r.origin_city, r.destination_city
              Buffers: shared hit=1041
              ->  Gather Merge  (cost=3671.87..3672.78 rows=8 width=126) (actual time=100.991..130.342 rows=40.00 loops=1)
                    Workers Planned: 1
                    Workers Launched: 1
                   

---
## Parte 4 — Creación de índices de optimización

Los 5 índices provistos por la escuela en `A2-03_optimization_indexes.sql`.
Cada índice tiene una justificación basada en las queries que beneficia.


In [15]:
conn = get_conn()
cur  = conn.cursor()

indices = [
    {
        'nombre': 'idx_trips_composite_joins',
        'sql': """
            CREATE INDEX IF NOT EXISTS idx_trips_composite_joins
            ON trips(vehicle_id, driver_id, route_id, departure_datetime)
            WHERE status = 'completed'
        """,
        'justificacion': 'JOINs frecuentes en B2, I1, I2, I4, C1, C3 — filtra + une en un solo índice parcial',
        'queries': ['B2', 'I1', 'I2', 'I4', 'C1', 'C3']
    },
    {
        'nombre': 'idx_deliveries_scheduled_datetime',
        'sql': """
            CREATE INDEX IF NOT EXISTS idx_deliveries_scheduled_datetime
            ON deliveries(scheduled_datetime, delivery_status)
            WHERE delivery_status = 'delivered'
        """,
        'justificacion': 'Filtra deliveries entregadas en B3, C1, C2 — índice parcial sobre 400k filas',
        'queries': ['B3', 'C1', 'C2']
    },
    {
        'nombre': 'idx_maintenance_vehicle_cost',
        'sql': """
            CREATE INDEX IF NOT EXISTS idx_maintenance_vehicle_cost
            ON maintenance(vehicle_id, cost)
        """,
        'justificacion': 'JOIN y SUM(cost) por vehículo en B1, I2, I5',
        'queries': ['B1', 'I2', 'I5']
    },
    {
        'nombre': 'idx_drivers_status_license',
        'sql': """
            CREATE INDEX IF NOT EXISTS idx_drivers_status_license
            ON drivers(status, license_expiry)
            WHERE status = 'active'
        """,
        'justificacion': 'Filtros por conductores activos en I1, C1, C4',
        'queries': ['I1', 'C1', 'C4']
    },
    {
        'nombre': 'idx_routes_metrics',
        'sql': """
            CREATE INDEX IF NOT EXISTS idx_routes_metrics
            ON routes(route_id, distance_km, destination_city)
        """,
        'justificacion': 'JOIN y cálculo de distancias en B2, I2, I4, C3',
        'queries': ['B2', 'I2', 'I4', 'C3']
    },
]

print('🔧 Creando índices de optimización...\n')
for idx in indices:
    t0 = time.time()
    cur.execute(idx['sql'])
    conn.commit()
    ms = round((time.time() - t0) * 1000, 0)
    print(f'  ✅ {idx["nombre"]}')
    print(f'     Justificación: {idx["justificacion"]}')
    print(f'     Queries beneficiadas: {idx["queries"]}')
    print(f'     Tiempo de creación: {ms} ms\n')

# Actualizar estadísticas para que el planificador use los nuevos índices
cur.execute('ANALYZE vehicles; ANALYZE drivers; ANALYZE routes; ANALYZE trips; ANALYZE deliveries; ANALYZE maintenance;')
conn.commit()
cur.close()
conn.close()

print('\n✅ Todos los índices creados y estadísticas actualizadas')
print('   Ahora ejecuta la Parte 5 para medir la mejora')


🔧 Creando índices de optimización...

  ✅ idx_trips_composite_joins
     Justificación: JOINs frecuentes en B2, I1, I2, I4, C1, C3 — filtra + une en un solo índice parcial
     Queries beneficiadas: ['B2', 'I1', 'I2', 'I4', 'C1', 'C3']
     Tiempo de creación: 12.0 ms

  ✅ idx_deliveries_scheduled_datetime
     Justificación: Filtra deliveries entregadas en B3, C1, C2 — índice parcial sobre 400k filas
     Queries beneficiadas: ['B3', 'C1', 'C2']
     Tiempo de creación: 0.0 ms

  ✅ idx_maintenance_vehicle_cost
     Justificación: JOIN y SUM(cost) por vehículo en B1, I2, I5
     Queries beneficiadas: ['B1', 'I2', 'I5']
     Tiempo de creación: 1.0 ms

  ✅ idx_drivers_status_license
     Justificación: Filtros por conductores activos en I1, C1, C4
     Queries beneficiadas: ['I1', 'C1', 'C4']
     Tiempo de creación: 1.0 ms

  ✅ idx_routes_metrics
     Justificación: JOIN y cálculo de distancias en B2, I2, I4, C3
     Queries beneficiadas: ['B2', 'I2', 'I4', 'C3']
     Tiempo de creació

---
## Parte 5 — EXPLAIN ANALYZE después de los índices

Mismas queries que en la Parte 3, ahora con los índices activos.
PostgreSQL debería elegir `Index Scan` en lugar de `Seq Scan` para las tablas grandes.

In [16]:
tiempos_despues = {}

print('⏱  Midiendo tiempos DESPUÉS de crear índices...\n')

print('── B2 ──')
tiempos_despues['B2'] = run_explain(SQL_B2, 'B2 después')

print('\n── I3 ──')
tiempos_despues['I3'] = run_explain(SQL_I3, 'I3 después')

print('\n── C1 ──')
tiempos_despues['C1'] = run_explain(SQL_C1, 'C1 después')

print('\n── C2 ──')
tiempos_despues['C2'] = run_explain(SQL_C2, 'C2 después')

print('\n── I5 ──')
tiempos_despues['I5'] = run_explain(SQL_I5, 'I5 después')

print('\n✅ Tiempos post-índices registrados:')
for q, t in tiempos_despues.items():
    print(f'   {q}: {t} ms')


⏱  Midiendo tiempos DESPUÉS de crear índices...

── B2 ──

── EXPLAIN ANALYZE: B2 después ──
Limit  (cost=3673.10..3673.12 rows=5 width=126) (actual time=107.485..126.710 rows=10.00 loops=1)
  Buffers: shared hit=1044
  ->  Sort  (cost=3673.10..3673.12 rows=5 width=126) (actual time=107.484..126.707 rows=10.00 loops=1)
        Sort Key: (count(t.trip_id)) DESC
        Sort Method: quicksort  Memory: 27kB
        Buffers: shared hit=1044
        ->  Finalize GroupAggregate  (cost=3671.87..3673.05 rows=5 width=126) (actual time=107.368..126.662 rows=20.00 loops=1)
              Group Key: r.origin_city, r.destination_city
              Buffers: shared hit=1041
              ->  Gather Merge  (cost=3671.87..3672.78 rows=8 width=126) (actual time=107.325..126.570 rows=40.00 loops=1)
                    Workers Planned: 1
                    Workers Launched: 1
                    Buffers: shared hit=1041
                    ->  Sort  (cost=2671.86..2671.87 rows=5 width=126) (actual time=76

---
## Parte 6 — Reporte comparativo de mejora

Calcula el porcentaje de mejora por query y genera el reporte final del Avance 2.

In [17]:
print('\n📊 REPORTE COMPARATIVO DE OPTIMIZACIÓN')
print('=' * 60)
print(f'  {"Query":<6} {"Antes (ms)":>12} {"Después (ms)":>13} {"Mejora":>10} {"Objetivo":>10}')
print('-' * 60)

resultados = []
for q in tiempos_antes:
    antes   = tiempos_antes[q]
    despues = tiempos_despues.get(q)
    if antes and despues:
        mejora_pct = round((antes - despues) / antes * 100, 1)
        cumple     = '✅' if mejora_pct >= 20 else '⚠️'
        print(f'  {q:<6} {antes:>12.1f} {despues:>13.1f} {mejora_pct:>9.1f}% {cumple:>10}')
        resultados.append({
            'query':       q,
            'antes_ms':    antes,
            'despues_ms':  despues,
            'mejora_pct':  mejora_pct,
            'cumple_20pct': mejora_pct >= 20
        })

print('=' * 60)

if resultados:
    prom = round(sum(r['mejora_pct'] for r in resultados) / len(resultados), 1)
    n_ok = sum(1 for r in resultados if r['cumple_20pct'])
    print(f'\n  Mejora promedio:          {prom}%')
    print(f'  Queries con mejora ≥20%:  {n_ok}/{len(resultados)}')
    print(f'  Objetivo del proyecto:    ≥20% de mejora')
    print(f'  Resultado:                {"✅ CUMPLIDO" if prom >= 20 else "⚠️ REVISAR"}')

    # Guardar reporte en JSON
    reporte = {
        'fecha':            datetime.now().isoformat(),
        'indices_creados':  5,
        'queries_medidas':  len(resultados),
        'mejora_promedio_pct': prom,
        'objetivo_cumplido': prom >= 20,
        'detalle':          resultados
    }
    with open('avance2_optimization_report.json', 'w') as f:
        json.dump(reporte, f, indent=2)
    print('\n  💾 Reporte guardado en avance2_optimization_report.json')



📊 REPORTE COMPARATIVO DE OPTIMIZACIÓN
  Query    Antes (ms)  Después (ms)     Mejora   Objetivo
------------------------------------------------------------
  B2            130.7         126.9       2.9%         ⚠️
  I3           1207.6         938.8      22.3%          ✅
  C1           2743.9        1931.2      29.6%          ✅
  C2           1149.9         781.1      32.1%          ✅
  I5             15.9           8.4      47.1%          ✅

  Mejora promedio:          26.8%
  Queries con mejora ≥20%:  4/5
  Objetivo del proyecto:    ≥20% de mejora
  Resultado:                ✅ CUMPLIDO

  💾 Reporte guardado en avance2_optimization_report.json


---
## Parte 7 — Vista Materializada

Una vista materializada guarda el **resultado** de una query en disco.
Se consulta como tabla normal pero es mucho más rápida porque ya está pre-calculada.
Se actualiza con `REFRESH MATERIALIZED VIEW` cuando llegan datos nuevos.

In [18]:
conn = get_conn()
cur  = conn.cursor()

# Crear vista materializada del reporte mensual (Query I3)
cur.execute("""
    DROP MATERIALIZED VIEW IF EXISTS mv_resumen_mensual;
    CREATE MATERIALIZED VIEW mv_resumen_mensual AS
    SELECT
        EXTRACT(YEAR  FROM t.departure_datetime)::INT  AS anio,
        EXTRACT(MONTH FROM t.departure_datetime)::INT  AS mes,
        COUNT(t.trip_id)                                AS total_viajes,
        COUNT(CASE WHEN t.status = 'completed'  THEN 1 END) AS completados,
        COUNT(CASE WHEN t.status = 'in_progress' THEN 1 END) AS en_progreso,
        ROUND(COUNT(CASE WHEN t.status='completed' THEN 1 END)*100.0
              / NULLIF(COUNT(t.trip_id),0), 1)          AS tasa_exito_pct,
        ROUND(SUM(t.fuel_consumed_liters), 0)           AS combustible_litros
    FROM trips t
    GROUP BY
        EXTRACT(YEAR  FROM t.departure_datetime),
        EXTRACT(MONTH FROM t.departure_datetime)
    ORDER BY anio, mes
""")
conn.commit()

print('✅ Vista materializada mv_resumen_mensual creada\n')

# Comparar velocidad: query directa vs vista materializada
print('⏱  Velocidad: query directa vs vista materializada')
t0  = time.time()
pd.read_sql_query('SELECT * FROM trips LIMIT 0', conn)  # warm up
t1  = time.time()
df_directo = pd.read_sql_query(SQL_I3, conn)
ms_directo = round((time.time() - t1) * 1000, 1)

t2 = time.time()
df_vista = pd.read_sql_query('SELECT * FROM mv_resumen_mensual', conn)
ms_vista  = round((time.time() - t2) * 1000, 1)

print(f'  Query I3 directa:          {ms_directo} ms')
print(f'  Vista materializada:        {ms_vista} ms')
if ms_directo > 0:
    mejora = round((ms_directo - ms_vista) / ms_directo * 100, 1)
    print(f'  Mejora con vista:           {mejora}%')

print('\nContenido de la vista:')
display(df_vista)

cur.close()
conn.close()


✅ Vista materializada mv_resumen_mensual creada

⏱  Velocidad: query directa vs vista materializada


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\3330328162.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('SELECT * FROM trips LIMIT 0', conn)  # warm up
C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\3330328162.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_directo = pd.read_sql_query(SQL_I3, conn)


  Query I3 directa:          686.9 ms
  Vista materializada:        5.7 ms
  Mejora con vista:           99.2%

Contenido de la vista:


C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\3330328162.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vista = pd.read_sql_query('SELECT * FROM mv_resumen_mensual', conn)


,anio,mes,total_viajes,completados,en_progreso,tasa_exito_pct,combustible_litros
0,2024,6,4020,4020,0,100.0,612754.0
1,2024,7,4464,4464,0,100.0,683385.0
2,2024,8,4464,4464,0,100.0,667669.0
3,2024,9,4320,4320,0,100.0,663832.0
4,2024,10,4464,4464,0,100.0,666043.0
5,2024,11,4320,4320,0,100.0,651320.0
6,2024,12,4464,4464,0,100.0,663539.0
7,2025,1,4464,4464,0,100.0,684211.0
8,2025,2,4032,4032,0,100.0,617944.0
9,2025,3,4464,4464,0,100.0,683087.0


---
## Parte 8 — Verificación de todos los índices activos

In [19]:
conn = get_conn()
df_idx = pd.read_sql_query("""
    SELECT
        tablename   AS tabla,
        indexname   AS indice,
        indexdef    AS definicion
    FROM pg_indexes
    WHERE schemaname = 'public'
    ORDER BY tablename, indexname
""", conn)
conn.close()

print(f'Total de índices en la BD: {len(df_idx)}\n')
display(df_idx)


Total de índices en la BD: 20



C:\Users\Ejecutivo-07\AppData\Local\Temp\ipykernel_10092\1829980422.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_idx = pd.read_sql_query("""


,tabla,indice,definicion
0,courses,courses_pkey,CREATE UNIQUE INDEX courses_pkey ON public.cou...
1,deliveries,deliveries_pkey,CREATE UNIQUE INDEX deliveries_pkey ON public....
2,deliveries,deliveries_tracking_number_key,CREATE UNIQUE INDEX deliveries_tracking_number...
3,deliveries,idx_deliveries_scheduled_datetime,CREATE INDEX idx_deliveries_scheduled_datetime...
4,deliveries,idx_deliveries_status,CREATE INDEX idx_deliveries_status ON public.d...
5,drivers,drivers_employee_code_key,CREATE UNIQUE INDEX drivers_employee_code_key ...
6,drivers,drivers_license_number_key,CREATE UNIQUE INDEX drivers_license_number_key...
7,drivers,drivers_pkey,CREATE UNIQUE INDEX drivers_pkey ON public.dri...
8,drivers,idx_drivers_status_license,CREATE INDEX idx_drivers_status_license ON pub...
9,maintenance,idx_maintenance_vehicle_cost,CREATE INDEX idx_maintenance_vehicle_cost ON p...
